<a href="https://colab.research.google.com/github/AlvaroAla/TE-IA/blob/main/GSI073_aula0_luong_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preparação dos dados

Esta tarefa é inverter sequências de caracteres. Exemplo: **aabcd** em **dcbaa**.


In [ ]:
import torch
import torch.nn as nn
import random
import torch.nn.functional as F

chars = list("abcd ")
vocab = {ch: i for i, ch in enumerate(chars)} # Cada letra, ganha um número
inv_vocab = {i: ch for ch, i in vocab.items()}# Tabela de decodificação
vocab_size = len(vocab)

def encode(s): # Codifica letras em números
    return torch.tensor([vocab[c] for c in s], dtype=torch.long)

def decode(t): # Decodifica números em letras
    return ''.join(inv_vocab[int(x)] for x in t)

def random_seq(n=5): # Cria novas sequências
    return ''.join(random.choice(chars[:-1]) for _ in range(n))

# Gerar dados
pairs = [(encode(s), encode(s[::-1])) for s in [random_seq() for _ in range(50000)]]

max_len = max(len(x) for x, _ in pairs) # pega maior sequência

def pad(x):  # Preenche conjunto de dados em pad no último índice
    return torch.cat([x, torch.tensor([vocab[' ']] * (max_len - len(x)))], dim=0)

inputs = torch.stack([pad(x) for x, _ in pairs])
targets = torch.stack([pad(y) for _, y in pairs])

train_ds = torch.utils.data.TensorDataset(inputs, targets)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Veja um par

In [ ]:
print(pairs[1])

# Definição do modelo Seq2Seq com GRU

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)

    def forward(self, x):
        x = self.embed(x)
        outputs, h = self.gru(x)
        return outputs, h   # <--- ESSENCIAL

In [ ]:
class LuongAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, decoder_hidden, encoder_outputs):
        """
        decoder_hidden: (B, 1, H)
        encoder_outputs: (B, S, H)

        Retorna:
          context: (B, 1, H)
          attn_weights: (B, 1, S)
        """

        # score = h_t · h_s^T
        # (B, 1, H) x (B, H, S) -> (B, 1, S)
        attn_scores = torch.bmm(decoder_hidden, encoder_outputs.transpose(1, 2))

        attn_weights = F.softmax(attn_scores, dim=-1)  # normaliza nos steps da source

        # context = soma ponderada
        # (B, 1, S) x (B, S, H) -> (B, 1, H)
        context = torch.bmm(attn_weights, encoder_outputs)

        return context, attn_weights

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)
        self.attn = LuongAttention()

        # Luong concat: concatena hidden + context
        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x, h, encoder_outputs):
        """
        x: tokens anteriores corretos  (B, T)
        h: estado inicial do decoder   (1, B, H)
        encoder_outputs: todos os h_s  (B, S, H)
        """
        x = self.embed(x)  # (B, T, E)

        outputs = []
        seq_len = x.size(1)
        hidden = h

        for t in range(seq_len):
            inp = x[:, t:t+1]  # (B, 1, E)

            out_t, hidden = self.gru(inp, hidden)   # out_t: (B,1,H)

            # Atenção
            context, attn_w = self.attn(out_t, encoder_outputs)

            # concatenação [out_t ; context]
            combined = torch.cat([out_t, context], dim=-1)

            logits = self.fc(combined)  # (B,1,V)
            outputs.append(logits)

        outputs = torch.cat(outputs, dim=1)  # (B, T, V)
        return outputs, hidden


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        encoder_outputs, h = self.encoder(src)
        logits, _ = self.decoder(tgt[:, :-1], h, encoder_outputs)
        return logits

# Código para usar o modelo treinado: inferência

In [ ]:
def decode_step(decoder, token, h, encoder_outputs):
    """
    Executa um passo de decodificação:
    - token: tensor (B,1)
    - h: estado oculto do decoder (1,B,H)
    - encoder_outputs: (B,S,H)
    """
    logits, h = decoder(token, h, encoder_outputs)  # (B,1,V)
    next_token = logits[:, -1, :].argmax(-1, keepdim=True)  # (B,1)
    return next_token, h


def predict(model, seq, max_len=10):
    model.eval()
    with torch.no_grad():
        # codifica entrada
        src = pad(encode(seq)).unsqueeze(0).to(device, dtype=torch.long)

        # encoder agora retorna (encoder_outputs, h)
        encoder_outputs, h = model.encoder(src)

        # token inicial (ex: espaço ou <sos>)
        token = torch.tensor([[vocab[' ']]], dtype=torch.long, device=device)

        seq_invertida = []
        for _ in range(max_len):
            token, h = decode_step(model.decoder, token, h, encoder_outputs)
            seq_invertida.append(token.item())

        return decode(seq_invertida)


# Preparação para treino

In [ ]:
emb_size = 32
hidden_size = 64
encoder = Encoder(vocab_size, emb_size, hidden_size)
decoder = Decoder(vocab_size, emb_size, hidden_size)
model = Seq2Seq(encoder, decoder).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=vocab[' ']) # ignora o pad: " "
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# Execução do treino

In [ ]:
for epoch in range(10):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device, dtype=torch.long), yb.to(device, dtype=torch.long)
        opt.zero_grad()
        logits = model(xb, yb)
        loss = loss_fn(logits.reshape(-1, vocab_size), yb[:, 1:].reshape(-1))
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_dl):.4f}")

# Vamos testar

In [ ]:
for _ in range(10):
    s = random_seq()
    pred = predict(model, s, max_len=len(s))
    print(f"{s} -> {pred}")


# Exercício
Compare o resultado do uso do encoder-decoder com atenção com o encoder-decoder sem atenção.

##Resposta do Exercicio

In [ ]:
class DecoderNoAttention(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h, encoder_outputs=None):
        x = self.embed(x)  # (B,T,E)
        out, h = self.gru(x, h)  # (B,T,H)
        logits = self.fc(out)    # (B,T,V)
        return logits, h


In [ ]:
class Seq2SeqNoAtt(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        _, h = self.encoder(src)
        logits, _ = self.decoder(tgt[:, :-1], h, None)
        return logits


In [ ]:
# Modelo COM atenção
encoder_att = Encoder(vocab_size, emb_size, hidden_size)
decoder_att = Decoder(vocab_size, emb_size, hidden_size)
model_att = Seq2Seq(encoder_att, decoder_att).to(device)

# Modelo SEM atenção
encoder_no = Encoder(vocab_size, emb_size, hidden_size)
decoder_no = DecoderNoAttention(vocab_size, emb_size, hidden_size)
model_no = Seq2SeqNoAtt(encoder_no, decoder_no).to(device)


In [ ]:
def train_model(model, train_dl, epochs=5):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss(ignore_index=vocab[' '])

    for epoch in range(epochs):
        model.train()
        total = 0
        for xb, yb in train_dl:

            xb = xb.to(device, dtype=torch.long)
            yb = yb.to(device, dtype=torch.long)

            opt.zero_grad()
            logits = model(xb, yb)

            loss = loss_fn(
                logits.reshape(-1, vocab_size),
                yb[:, 1:].reshape(-1)
            )
            loss.backward()
            opt.step()
            total += loss.item()

        print(f"[{model.__class__.__name__}] Epoch {epoch+1}: loss={total/len(train_dl):.4f}")


In [ ]:
print("Treinando COM atenção")
train_model(model_att, train_dl, epochs=5)

print("\nTreinando SEM atenção")
train_model(model_no, train_dl, epochs=5)


In [ ]:
def predict_no(model, seq):
    model.eval()
    with torch.no_grad():

        # entrada do encoder (sempre long!)
        src = pad(encode(seq)).unsqueeze(0).to(device, dtype=torch.long)

        _, h = model.encoder(src)

        # token inicial
        token = torch.tensor([[vocab[' ']]], device=device, dtype=torch.long)

        out = []
        for _ in range(len(seq)):

            # GARANTIR QUE token É LONG, SEMPRE
            token = token.to(device, dtype=torch.long)

            # passa pelo decoder
            logits, h = model.decoder(token, h, None)

            # argmax vira LONG obrigatoriamente
            next_token = logits[:, -1, :].argmax(-1, keepdim=True).long()

            token = next_token
            out.append(token.item())

        return decode(out)


In [ ]:
print("\n=== COMPARAÇÃO FINAL ===\n")

for _ in range(10):
    s = random_seq()
    p_att = predict_att(model_att, s)
    p_no  = predict_no(model_no,  s)
    print(f"Entrada : {s}")
    print(f"Com ATT: {p_att}")
    print(f"Sem ATT: {p_no}")
    print("-" * 40)


##Comparação do modelo com atenção vs modelo LSTM sem atenção

In [ ]:
class DecoderLSTMNoAttention(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.lstm = nn.LSTM(emb_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h, encoder_outputs=None):
        x = self.embed(x)  # (B,T,E)
        out, h = self.lstm(x, h)  # (B,T,H)
        logits = self.fc(out)     # (B,T,V)
        return logits, h


In [ ]:
class Seq2SeqLSTMNoAtt(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        _, h = self.encoder(src)

        # GRU retorna apenas h → precisamos transformar em h e c para o LSTM
        # Inicializar c como zeros
        h0 = h
        c0 = torch.zeros_like(h)
        lstm_state = (h0, c0)

        logits, _ = self.decoder(tgt[:, :-1], lstm_state)
        return logits


In [ ]:
encoder_lstm = Encoder(vocab_size, emb_size, hidden_size)   # GRU encoder
decoder_lstm = DecoderLSTMNoAttention(vocab_size, emb_size, hidden_size)
model_lstm = Seq2SeqLSTMNoAtt(encoder_lstm, decoder_lstm).to(device)


In [ ]:
def predict_lstm(model, seq):
    model.eval()
    with torch.no_grad():

        src = pad(encode(seq)).unsqueeze(0).to(device, dtype=torch.long)

        # Encoder GRU
        _, h = model.encoder(src)

        # Estado inicial do LSTM
        h0 = h
        c0 = torch.zeros_like(h)
        lstm_state = (h0, c0)

        token = torch.tensor([[vocab[' ']]], device=device, dtype=torch.long)

        out = []
        for _ in range(len(seq)):

            token = token.to(device, dtype=torch.long)

            logits, lstm_state = model.decoder(token, lstm_state)
            next_token = logits[:, -1, :].argmax(-1, keepdim=True).long()

            token = next_token
            out.append(token.item())

        return decode(out)


In [ ]:
print("\nTreinando LSTM sem atenção")
train_model(model_lstm, train_dl, epochs=5)


In [ ]:
print("\n=== COMPARAÇÃO FINAL ===\n")

for _ in range(10):
    s = random_seq()

    p_att  = predict_att(model_att,  s)   # GRU + Atenção
    p_lstm = predict_lstm(model_lstm, s)  # LSTM sem Atenção

    print(f"Entrada : {s}")
    print(f"Com ATT (GRU+Att): {p_att}")
    print(f"LSTM sem Att     : {p_lstm}")
    print("-" * 40)
